# Memory jednej rozmowy: najpierw Python, potem checkpointer

W poprzedniej lekcji pętla wysyłała do agenta tylko nowe pytanie, więc model nie dostawał poprzednich tur. Tutaj zachowamy historię **jednej rozmowy w pamięci kernela**. Nie zapisujemy jej do pliku. Po restarcie kernela zaczynamy od nowa.

Najpierw sami prowadzimy zwykły słownik Pythona. Potem zobaczymy, jak tę samą pracę przejmuje checkpointer agenta. Do wywołania modelu używamy wspólnej fabryki projektu `get_model()`.


In [ ]:
# Tylko importy i minimalne ustawienie ścieżki projektu.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from model_config import get_model


In [ ]:
model = get_model()


## 1. Ręcznie: słownik rośnie po każdej turze

`conversation` jest zwykłym słownikiem. W `messages` trzymamy komunikat systemowy oraz kolejne pytania i odpowiedzi. Przed każdym wywołaniem **my** przekazujemy modelowi całą dotychczasową historię razem z nowym pytaniem. Sam model nie pamięta wcześniejszego wywołania.


In [ ]:
conversation = {
    "messages": [
        {"role": "system", "content": "Odpowiadaj krótko po polsku."},
    ]
}
print(conversation)


In [ ]:
def ask_manually(question: str) -> None:
    user_message = {"role": "user", "content": question}
    messages_for_model = conversation["messages"] + [user_message]
    answer = model.invoke(messages_for_model).content

    conversation["messages"].append(user_message)
    conversation["messages"].append({"role": "assistant", "content": answer})

    print("Odpowiedź:", answer)
    print("Liczba komunikatów w słowniku:", len(conversation["messages"]))
    print(conversation)


In [ ]:
ask_manually("Mam na imię Ola. Zapamiętaj moje imię.")


In [ ]:
ask_manually("Jak mam na imię?")


Po pierwszej turze słownik zawiera 3 komunikaty: system, użytkownik, asystent. Po drugiej zawiera 5. Drugie wywołanie dostało wszystkie 3 wcześniejsze komunikaty, dlatego może odpowiedzieć na pytanie o imię. Możesz wywołać `ask_manually(...)` ponownie, aby dodać kolejną turę.

## 2. LangChain: checkpointer przechowuje historię

`create_agent` z `InMemorySaver` dopisuje historię automatycznie. Przy każdym wywołaniu podajemy tylko **nowe** pytanie. Stały `thread_id` wskazuje tę jedną rozmowę; nie ma tutaj wybierania innych rozmów ani zapisu na dysk.


In [ ]:
checkpointer = InMemorySaver()
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="Odpowiadaj krótko po polsku.",
    checkpointer=checkpointer,
)
config = {"configurable": {"thread_id": "jedna-rozmowa"}}


In [ ]:
first = agent.invoke(
    {"messages": [{"role": "user", "content": "Mam na imię Ola. Zapamiętaj moje imię."}]},
    config=config,
)
print("Odpowiedź:", first["messages"][-1].content)
print("Liczba komunikatów w checkpointcie:", len(first["messages"]))


In [ ]:
second = agent.invoke(
    {"messages": [{"role": "user", "content": "Jak mam na imię?"}]},
    config=config,
)
print("Odpowiedź:", second["messages"][-1].content)
print("Liczba komunikatów w checkpointcie:", len(second["messages"]))


W obu wariantach kolejne pytanie trafia do modelu z wcześniejszym kontekstem. Różnica: w pierwszym wariancie uzupełniamy słownik i przekazujemy całą historię sami; w drugim checkpointer robi to za agenta. `InMemorySaver` działa tylko w bieżącym procesie. Następna lekcja dodaje zapis, odczyt i zmianę ID rozmowy.
